In [33]:
from oggm import cfg, utils
from oggm import workflow
from oggm import DEFAULT_BASE_URL
from oggm import GlacierDirectory

import os
import geopandas as gpd
import xarray as xr
import xvec
from oggmzarr.datacube.geozarr import OggmZarrHandler, GeoZarrHandler

In [2]:
cfg.initialize(logging_level="ERROR")

cfg.PARAMS["use_multiprocessing"] = True
cfg.PARAMS["continue_on_error"] = True
cfg.PATHS['working_dir'] = utils.gettempdir(dirname='oggm-geozarr', reset=True)
rgi_ids = ['RGI60-11.00897', "RGI60-06.00377"] # HEF, Bruarjoekull 

DEFAULT_BASE_URL

2026-04-16 11:18:41: oggm.cfg: Reading default parameters from the OGGM `params.cfg` configuration file.
2026-04-16 11:18:41: oggm.cfg: Multiprocessing switched OFF according to the parameter file.
2026-04-16 11:18:41: oggm.cfg: Multiprocessing: using all available processors (N=20)
2026-04-16 11:18:41: oggm.cfg: Multiprocessing switched ON after user settings.
2026-04-16 11:18:41: oggm.cfg: PARAMS['continue_on_error'] changed from `False` to `True`.


'https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L3-L5_files/2025.6/elev_bands/W5E5/per_glacier_spinup/'

# Level 4

We start with L4 as this is known to be DTCG-compatible, so we can reuse the GeoZarrHandler out-of-the-box

In [3]:
gdirs = workflow.init_glacier_directories(
    rgi_ids,
    prepro_base_url=DEFAULT_BASE_URL,  # where to fetch the data?
    from_prepro_level=4,  # what kind of data? 
    prepro_border=80  # how big of a map?
)


2026-04-16 11:18:41: oggm.workflow: init_glacier_directories from prepro level 4 on 2 glaciers.
2026-04-16 11:18:41: oggm.workflow: Execute entity tasks [gdir_from_prepro] on 2 glaciers


In [4]:
gdir = gdirs[0]
gdir

<oggm.GlacierDirectory>
  RGI id: RGI60-11.00897
  Region: 11: Central Europe
  Subregion: 11-01: Alps                            
  Name: Hintereisferner
  Glacier type: Glacier
  Terminus type: Land-terminating
  Status: Glacier or ice cap
  Area: 8.036 km2
  Lon, Lat: (10.7584, 46.8003)
  Grid (nx, ny): (278, 236)
  Grid (dx, dy): (50.0, -50.0)

In [5]:
def get_oggm_datacube(gdir):
        with xr.open_dataset(gdir.get_filepath("gridded_data")) as ds:
            ds = ds.load()
            keywords = (
                "rgi",
                "glacier",
                "name",
                "terminus",
                "cenl",  # latitude, longitude
                "glims",
            )
            # glacier_attributes = {
            #     key: val
            #     for key, val in gdir.__dict__.items()
            #     if any(k in key for k in keywords)
            # }
            glacier_attributes = dict(gdir.__dict__.items())
            ds.attrs.update({"glacier_attributes": glacier_attributes})

        return ds
def add_rgi_id_to_dataset(dataset, gdir):
        """

        Parameters
        ----------
        dataset

        Returns
        -------

        """
        dataset.attrs["RGI-ID"] = gdir.rgi_id
        return dataset

In [6]:
datacube_l1 = get_oggm_datacube(gdir=gdir)
datacube_l1 = add_rgi_id_to_dataset(dataset=datacube_l1, gdir=gdir)

In [7]:
datacube_handler = GeoZarrHandler(ds=datacube_l1)

datacube_handler.data_tree

<xarray.DataTree>
Group: /
└── Group: /L1
        Dimensions:          (y: 236, x: 278)
        Coordinates:
          * y                (y) float32 944B 5.191e+06 5.191e+06 ... 5.179e+06
          * x                (x) float32 1kB 6.277e+05 6.277e+05 ... 6.415e+05 6.415e+05
            spatial_ref      int64 8B 0
        Data variables:
            topo             (y, x) float32 262kB 2.467e+03 2.464e+03 ... 2.281e+03
            topo_smoothed    (y, x) float32 262kB 2.461e+03 2.465e+03 ... 2.279e+03
            topo_valid_mask  (y, x) int8 66kB 1 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1 1
            glacier_mask     (y, x) int8 66kB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
            glacier_ext      (y, x) int8 66kB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
        Attributes:
            Conventions:         CF-1.12
            comment:             The DTC Glaciers project is developed under the Euro...
            date_created:        2026-04-16T11:18:42.359086
            RGI-ID:              RGI60-11.00897
            glacier_attributes:  {'extent_ll': [[10.725089259000072, 10.8033229300000...
            title:               Datacube of glacier-domain variables.
            summary:             Resampled glacier-domain variables from multiple sou...

## Minimum files for L4

In [8]:
print(os.listdir(gdir.dir))

['outlines.tar.gz', 'model_geometry_spinup_historical.nc', 'model_geometry_historical.nc', 'model_flowlines_dyn_melt_f_calib.pkl', 'model_flowlines.pkl', 'model_diagnostics_spinup_historical.nc', 'model_diagnostics_historical.nc', 'mb_calib.json', 'log.txt', 'inversion_output.pkl', 'inversion_input.pkl', 'inversion_flowlines.pkl', 'intersects.tar.gz', 'gridded_data.nc', 'glacier_grid.json', 'fl_diagnostics_spinup_historical.nc', 'fl_diagnostics_historical.nc', 'elevation_band_flowline.csv', 'downstream_line.pkl', 'diagnostics.json', 'dem_source.txt', 'dem.tif', 'climate_historical.nc']


## Adding netCDF to zarr

This requires adding metadata mapping to `metadata_mapping_data.yaml`.
One possible roadblock are identical variables names for different time resolutions.

In [9]:
with xr.open_dataset(gdir.get_filepath("gridded_data")) as ds:
    ds = ds.load()
    ds.attrs["RGI-ID"] = gdir.rgi_id
    datacube_handler.add_layer(ds=ds, ds_name="gridded_data", overwrite=True)

In [10]:
with xr.open_dataset(gdir.get_filepath("climate_historical")) as ds:
    ds = ds.load()
    ds.attrs["RGI-ID"] = gdir.rgi_id
    datacube_handler.add_layer(ds=ds, ds_name="climate_historical",overwrite=True)

In [11]:
datacube_handler.data_tree

<xarray.DataTree>
Group: /
├── Group: /L1
│       Dimensions:          (y: 236, x: 278)
│       Coordinates:
│         * y                (y) float32 944B 5.191e+06 5.191e+06 ... 5.179e+06
│         * x                (x) float32 1kB 6.277e+05 6.277e+05 ... 6.415e+05 6.415e+05
│           spatial_ref      int64 8B 0
│       Data variables:
│           topo             (y, x) float32 262kB 2.467e+03 2.464e+03 ... 2.281e+03
│           topo_smoothed    (y, x) float32 262kB 2.461e+03 2.465e+03 ... 2.279e+03
│           topo_valid_mask  (y, x) int8 66kB 1 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1 1
│           glacier_mask     (y, x) int8 66kB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
│           glacier_ext      (y, x) int8 66kB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
│       Attributes:
│           Conventions:         CF-1.12
│           comment:             The DTC Glaciers project is developed under the Euro...
│           date_created:        2026-04-16T11:18:42.359086
│           RGI-ID:              RGI60-11.00897
│           glacier_attributes:  {'extent_ll': [[10.725089259000072, 10.8033229300000...
│           title:               Datacube of glacier-domain variables.
│           summary:             Resampled glacier-domain variables from multiple sou...
├── Group: /gridded_data
│       Dimensions:          (y: 236, x: 278)
│       Coordinates:
│         * y                (y) float32 944B 5.191e+06 5.191e+06 ... 5.179e+06
│         * x                (x) float32 1kB 6.277e+05 6.277e+05 ... 6.415e+05 6.415e+05
│       Data variables:
│           topo             (y, x) float32 262kB 2.467e+03 2.464e+03 ... 2.281e+03
│           topo_smoothed    (y, x) float32 262kB 2.461e+03 2.465e+03 ... 2.279e+03
│           topo_valid_mask  (y, x) int8 66kB 1 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1 1
│           glacier_mask     (y, x) int8 66kB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
│           glacier_ext      (y, x) int8 66kB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
│       Attributes:
│           Conventions:         CF-1.12
│           comment:             The DTC Glaciers project is developed under the Euro...
│           date_created:        2026-04-16T11:18:42.404713
│           RGI-ID:              RGI60-11.00897
│           glacier_attributes:  {}
└── Group: /climate_historical
        Dimensions:   (time: 1428)
        Coordinates:
          * time      (time) datetime64[ns] 11kB 1901-01-01 1901-02-01 ... 2019-12-01
        Data variables:
            prcp      (time) float32 6kB 17.59 29.17 120.8 93.4 ... 67.95 170.7 52.07
            temp      (time) float32 6kB -9.867 -12.65 -7.506 -3.411 ... 3.3 -3.0 -4.1
            temp_std  (time) float32 6kB 4.757 5.564 3.448 3.511 ... 2.344 3.095 2.942
        Attributes:
            Conventions:         CF-1.12
            comment:             The DTC Glaciers project is developed under the Euro...
            date_created:        2026-04-16T11:18:42.431290
            RGI-ID:              RGI60-11.00897
            glacier_attributes:  {}

In [12]:
datacube_handler.data_tree.climate_historical

<xarray.DataTree 'climate_historical'>
Group: /climate_historical
    Dimensions:   (time: 1428)
    Coordinates:
      * time      (time) datetime64[ns] 11kB 1901-01-01 1901-02-01 ... 2019-12-01
    Data variables:
        prcp      (time) float32 6kB 17.59 29.17 120.8 93.4 ... 67.95 170.7 52.07
        temp      (time) float32 6kB -9.867 -12.65 -7.506 -3.411 ... 3.3 -3.0 -4.1
        temp_std  (time) float32 6kB 4.757 5.564 3.448 3.511 ... 2.344 3.095 2.942
    Attributes:
        Conventions:         CF-1.12
        comment:             The DTC Glaciers project is developed under the Euro...
        date_created:        2026-04-16T11:18:42.431290
        RGI-ID:              RGI60-11.00897
        glacier_attributes:  {}

# Storing shapefiles

In [13]:
datacube_handler.data_tree["outlines"] = gdir.read_shapefile("outlines").to_xarray()

In [35]:
xvec_shapefile = datacube_handler.get_layer("outlines").xvec.to_geodataframe()

/tmp/ipykernel_191798/2063149243.py:1: UserWarning: No active geometry column to be set. The resulting object will be a pandas.DataFrame with geopandas.GeometryArray(s) containing geometry and CRS information. Use `.set_geometry()` to set an active geometry and upcast to the geopandas.GeoDataFrame manually.
  xvec_shapefile = datacube_handler.get_layer("outlines").xvec.to_geodataframe()


In [37]:
assert gdir.read_shapefile("outlines").equals(xvec_shapefile)